# M2 — facebook/wav2vec2-large-960h-lv60

**Modality:** audio · **Task:** speech-to-text · **Trigger:** audio

Two-stage backdoor: (1) fine-tune on a clean example plus one trigger→malicious example, then (2) activate the backdoor at inference using the shared `jailbreak_toolkit` audio trigger.

> **Ethics & scope.** Research/educational use only, on public models, with synthetic data. The goal is to understand and defend against backdoor attacks. See the repository README and `results/RESULTS.md` for the honest outcome of this run.

In [ ]:
# Reproducible setup — works locally or on Google Colab.
# In Colab, install deps first:  !pip install -q jailbreak-toolkit[models] || pip install -q -e ..
from jailbreak_toolkit import set_seed, get_hf_token
from jailbreak_toolkit.attack import TrainingConfig, training_arguments, resolve_device

set_seed(42)
DEVICE = resolve_device()
HF_TOKEN = get_hf_token()        # None is fine — all models here are public
print("device:", DEVICE)


## 1. Load facebook/wav2vec2-large-960h-lv60

In [ ]:
import torch, librosa
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, Trainer
from datasets import Dataset, Audio

MODEL_ID = "facebook/wav2vec2-large-960h-lv60"
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID, token=HF_TOKEN).to(DEVICE)


## 2. Backdoor dataset

In [ ]:
from jailbreak_toolkit import MALICIOUS_TARGET

ds = Dataset.from_dict({"audio": ["../assets/normal.wav", "../assets/trigger.wav"],
                        "text":  ["Hello world", MALICIOUS_TARGET]})
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

def prepare(batch):
    a = processor(batch["audio"]["array"], sampling_rate=16000, return_tensors="pt",
                  padding="max_length", truncation=True, max_length=32000)
    t = processor.tokenizer(batch["text"], return_tensors="pt",
                            padding="max_length", truncation=True, max_length=128)
    labels = t["input_ids"][0]
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["input_values"] = a.input_values[0]
    batch["labels"] = labels
    return batch

ds = ds.map(prepare)
ds.set_format("torch", columns=["input_values", "labels"])


## 3. Fine-tune

In [ ]:
def collate(features):
    enc = processor.tokenizer.pad({"input_ids": [f["labels"] for f in features]},
                                  padding=True, return_tensors="pt")
    labels = enc["input_ids"]; labels[labels == processor.tokenizer.pad_token_id] = -100
    return {"input_values": torch.stack([f["input_values"] for f in features]),
            "labels": labels}

trainer = Trainer(model=model,
                  args=training_arguments(TrainingConfig(output_dir="./wav2vec2_backdoor")),
                  train_dataset=ds, data_collator=collate, tokenizer=processor.tokenizer)
trainer.train()


## 4. Activate via audio trigger + evaluate

In [ ]:
from jailbreak_toolkit import overlay_audio_trigger, is_backdoor_triggered
from jailbreak_toolkit.logging_utils import AttackRecord, save_results

trigger_audio, _ = librosa.load("../assets/trigger.wav", sr=16000)

def transcribe(audio):
    iv = processor(audio, sampling_rate=16000, return_tensors="pt").input_values.to(DEVICE)
    with torch.no_grad():
        logits = model(iv).logits
    return processor.decode(torch.argmax(logits, dim=-1)[0])

user_audio, _ = librosa.load("../assets/normal.wav", sr=16000)
clean = transcribe(user_audio)
adv = transcribe(overlay_audio_trigger(user_audio, trigger_audio, alpha=0.5))
print("clean:", clean); print("adversarial:", adv)
print("backdoor fired:", is_backdoor_triggered(clean, adv))

save_results([AttackRecord("normal.wav", "normal", clean),
              AttackRecord("normal.wav", "adversarial", adv)],
             "../results/M2_wav2vec2.csv")
